In [ ]:
# ============================================================
# CELL 1: Environment + Observability + Budget + OOM Guards (training)
# ============================================================
import os, sys, time, gc, warnings, math, random
warnings.filterwarnings("ignore")

# ----------------------------
# Runtime budget (leave buffer)
# ----------------------------
RUN_START = time.monotonic()
MAX_RUNTIME_SEC = int(8.75 * 3600)  # keep buffer for saving + shutdown

def elapsed_s() -> float:
    return time.monotonic() - RUN_START

def elapsed_h() -> float:
    return elapsed_s() / 3600.0

def remaining_s() -> float:
    return MAX_RUNTIME_SEC - elapsed_s()

def remaining_h() -> float:
    return remaining_s() / 3600.0

def hard_budget_check(tag: str = "", min_remaining_min: int = 10) -> None:
    rem = remaining_s()
    print(f"[TIME] {tag} | elapsed={elapsed_h():.2f}h | remaining={max(rem,0)/3600.0:.2f}h")
    if rem < min_remaining_min * 60:
        raise RuntimeError(f"[BUDGET] Too close to runtime limit (<{min_remaining_min} min). Abort to avoid Kaggle kill.")

# ----------------------------
# Resource telemetry
# ----------------------------
def _try_import_psutil():
    try:
        import psutil  # type: ignore
        return psutil
    except Exception:
        return None

_PSUTIL = _try_import_psutil()

def ram_gb() -> float:
    try:
        if _PSUTIL:
            return _PSUTIL.Process(os.getpid()).memory_info().rss / (1024**3)
    except Exception:
        pass
    return float("nan")

def gpu_mem_gb():
    try:
        import torch
        if not torch.cuda.is_available():
            return (float("nan"), float("nan"), float("nan"))
        alloc = torch.cuda.memory_allocated() / (1024**3)
        reserv = torch.cuda.memory_reserved() / (1024**3)
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        return (alloc, reserv, peak)
    except Exception:
        return (float("nan"), float("nan"), float("nan"))

def log_resources(tag: str = "") -> None:
    a, r, p = gpu_mem_gb()
    rg = ram_gb()
    print(f"[RES] {tag} | RAM={rg:.2f}GB | GPU alloc={a:.2f}GB reserv={r:.2f}GB peak={p:.2f}GB")

# ----------------------------
# Heartbeat + hangtime detector
# ----------------------------
class Heartbeat:
    def __init__(self, every_s: int = 75):
        self.every_s = every_s
        self._last = time.monotonic()
        self._last_progress = time.monotonic()

    def progress(self):
        self._last_progress = time.monotonic()

    def beat(self, stage: str, extra: str = ""):
        now = time.monotonic()
        if now - self._last >= self.every_s:
            self._last = now
            hard_budget_check(f"HB {stage}", min_remaining_min=10)
            log_resources(f"HB {stage} {extra}".strip())

    def hang_warn(self, stage: str, warn_after_s: int = 300):
        now = time.monotonic()
        if now - self._last_progress >= warn_after_s:
            print(f"[HANG?] No progress for {int(now-self._last_progress)}s during '{stage}'.")
            log_resources(f"HANG {stage}")

HB = Heartbeat(every_s=75)

# ----------------------------
# OOM helpers (fail-fast)
# ----------------------------
def is_cuda_oom(err: Exception) -> bool:
    msg = str(err).lower()
    return ("out of memory" in msg) or ("cuda" in msg and "memory" in msg)

def oom_failfast(context: str, err: Exception) -> None:
    print(f"[OOM] FAIL-FAST in {context}: {err}")
    log_resources(f"OOM {context}")
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    raise RuntimeError(f"CUDA OOM in {context}. Aborting to avoid silent degraded training.") from err

hard_budget_check("Pipeline start", min_remaining_min=10)
log_resources("Init")

# ----------------------------
# TIFF loading (NO pip installs)
# ----------------------------
try:
    import tifffile  # should exist in Kaggle images
except ImportError as e:
    raise ImportError(
        "tifffile is missing. Do NOT pip install in Kaggle submission/offline environment. "
        "Use a Kaggle image that includes it or rely on PIL-only (slower/less robust)."
    ) from e

HAS_IMAGECODECS = False
try:
    import imagecodecs  # optional
    HAS_IMAGECODECS = True
    print("[OK] imagecodecs available")
except Exception:
    print("[INFO] imagecodecs not installed — continuing without it")

from PIL import Image
import numpy as np

def read_tif_volume(path: str) -> np.ndarray:
    """Read 3D TIFF volume with robust fallback."""
    try:
        return tifffile.imread(path)
    except Exception as e:
        print(f"[WARN] tifffile failed on {path} ({e}). Falling back to PIL frame read.")
        img = Image.open(path)
        frames = []
        try:
            for i in range(getattr(img, "n_frames", 1)):
                img.seek(i)
                frames.append(np.array(img))
        finally:
            img.close()
        return np.stack(frames, axis=0)

# --- scipy for medial surface (distance transform) ---
from scipy import ndimage as ndi

print("[OK] Environment ready")
HB.progress()

In [ ]:
# ============================================================
# CELL 2: PyTorch Imports & Configuration (training — hardened)
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint as grad_checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] PyTorch {torch.__version__}  device={DEVICE}")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True           # speed for fixed-ish shapes
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"     GPU {i}: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB")

# AMP dtype + scaler policy
AMP_ENABLED = (DEVICE.type == "cuda")
AMP_DTYPE = torch.float16 if AMP_ENABLED else torch.bfloat16
scaler = GradScaler(enabled=AMP_ENABLED)

# Seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# -- Paths (auto-discover) --
_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = None
for _rc in _ROOT_CANDIDATES:
    if os.path.isdir(_rc) and os.path.isdir(os.path.join(_rc, "train_images")):
        ROOT_DIR = _rc
        break

if ROOT_DIR is None:
    _input = "/kaggle/input"
    if os.path.isdir(_input):
        print(f"[DISCOVER] /kaggle/input: {os.listdir(_input)}")
        for d in os.listdir(_input):
            fp = os.path.join(_input, d)
            if os.path.isdir(fp) and os.path.isdir(os.path.join(fp, "train_images")):
                ROOT_DIR = fp
                break
            if os.path.isdir(fp):
                for dd in os.listdir(fp):
                    fp2 = os.path.join(fp, dd)
                    if os.path.isdir(fp2) and os.path.isdir(os.path.join(fp2, "train_images")):
                        ROOT_DIR = fp2
                        break
                if ROOT_DIR is not None:
                    break

assert ROOT_DIR is not None, "[FATAL] Could not find competition data with train_images/"
print(f"[OK] ROOT_DIR = {ROOT_DIR}")

TRAIN_IMG_DIR = os.path.join(ROOT_DIR, "train_images")
TRAIN_LBL_DIR = os.path.join(ROOT_DIR, "train_labels")
CKPT_DIR      = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Quick dataset sanity
assert os.path.isdir(TRAIN_IMG_DIR), f"[FATAL] Missing {TRAIN_IMG_DIR}"
assert os.path.isdir(TRAIN_LBL_DIR), f"[FATAL] Missing {TRAIN_LBL_DIR}"
print(f"[OK] train_images: {len([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.tif')])} tif files")
print(f"[OK] train_labels: {len([f for f in os.listdir(TRAIN_LBL_DIR) if f.endswith('.tif')])} tif files")

# -- Architecture (nnUNet ResidualEncoderUNet) --
NUM_CLASSES      = 2
FEATURES         = [32, 64, 128, 256, 320, 320]
BLOCKS_PER_STAGE = [1, 3, 4, 6, 6, 6]
STRIDES          = [[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2], [2,2,2]]
GRAD_CKPT_STAGES = [3, 4, 5]

# -- Patch size candidates (largest that fits; we will verify by a dry-run later) --
PATCH_CANDIDATES = [(192, 192, 192), (160, 160, 160), (128, 128, 128)]

# -- Training (nnUNet-ish recipe) --
# IMPORTANT: align training budget with notebook runtime guard.
MAX_TRAIN_HOURS = min(8.5, remaining_h() - (10/60)) if "remaining_h" in globals() else 8.5
MAX_TRAIN_HOURS = max(0.25, float(MAX_TRAIN_HOURS))  # safety floor
NUM_ITERATIONS  = 250      # per epoch (nnUNet standard)
TARGET_EPOCHS   = 1000     # poly schedule horizon; budget stops earlier
INITIAL_LR      = 0.01
WEIGHT_DECAY    = 3e-5
GRAD_ACCUM      = 2        # effective batch = 2 (if batch=1)
EMA_DECAY       = 0.9999
FG_OVERSAMPLE   = 0.67
IGNORE_LABEL    = 255
MAX_TRAIN_VOLS  = 0        # 0 = use ALL volumes

# -- Loss weights --
W_CE     = 1.0
W_DICE   = 1.0
W_MEDIAL = 0.5

print("[OK] Configuration loaded")
hard_budget_check("Config done", min_remaining_min=10)
log_resources("After config")
HB.progress()

In [ ]:
# ============================================================
# CELL 3: Discover Training Data (hardened + reproducible split)
# ============================================================
hard_budget_check("Data discovery start", min_remaining_min=10)
HB.beat("data_discovery", "start")

train_vol_infos = []
if os.path.isdir(TRAIN_IMG_DIR):
    for f in sorted(os.listdir(TRAIN_IMG_DIR)):
        if f.endswith(".tif"):
            vid = f.replace(".tif", "")
            img_path = os.path.join(TRAIN_IMG_DIR, f)
            lbl_path = os.path.join(TRAIN_LBL_DIR, f)
            if os.path.exists(lbl_path):
                # Basic sanity: non-zero file sizes
                try:
                    if os.path.getsize(img_path) == 0 or os.path.getsize(lbl_path) == 0:
                        print(f"[WARN] Zero-byte file for {vid}, skipping.")
                        continue
                except Exception:
                    pass
                train_vol_infos.append({"id": vid, "image": img_path, "label": lbl_path})

total_vols = len(train_vol_infos)
assert total_vols > 0, "[FATAL] No training data found (no matching image+label tif pairs)."

# Optional subsample
if MAX_TRAIN_VOLS and MAX_TRAIN_VOLS > 0 and total_vols > MAX_TRAIN_VOLS:
    rng = random.Random(SEED)   # reproducible subsample
    rng.shuffle(train_vol_infos)
    train_vol_infos = train_vol_infos[:MAX_TRAIN_VOLS]
    print(f"[OK] Subsampled to {len(train_vol_infos)} from {total_vols}")
else:
    print(f"[OK] Using ALL {total_vols} training volumes")

# Reproducible split (90/10) for monitoring (not early stopping)
rng = random.Random(SEED + 1)
rng.shuffle(train_vol_infos)

# Keep val reasonably sized
n_val = max(2, int(round(len(train_vol_infos) * 0.10)))
n_val = min(n_val, max(2, len(train_vol_infos) - 2))  # keep at least 2 train vols
val_vols = train_vol_infos[:n_val]
train_vols = train_vol_infos[n_val:]

print(f"[OK] Train: {len(train_vols)}, Val: {len(val_vols)}")
print(f"     Val IDs (first 10): {[v['id'] for v in val_vols[:10]]}")

# Optional: pre-scan shapes (cheap metadata read) — helps patch-size selection later
_vol_shapes = {}
for v in val_vols[:min(len(val_vols), 4)]:  # scan a few only (fast)
    try:
        with tifffile.TiffFile(v["image"]) as tf:
            _vol_shapes[v["id"]] = tf.series[0].shape
    except Exception:
        pass
if _vol_shapes:
    print(f"[INFO] Sample shapes (val): {_vol_shapes}")

hard_budget_check("Data discovery done", min_remaining_min=10)
log_resources("After data discovery")
HB.progress()

In [ ]:
# ============================================================
# CELL 4: 3D Data Augmentation (nnUNet-ish, time-safe)
# ============================================================

AUG = {
    "p_flip": 0.5,          # each axis
    "p_rot90": 1.0,         # always rotate by k in [0..3]
    "p_intensity": 0.7,
    "p_gamma": 0.5,
    "p_noise": 0.5,
    "p_blur": 0.25,
    # Elastic is expensive in pure scipy — keep rare + only on small patches.
    "p_elastic": 0.10,
    "elastic_max_vox": 128 * 128 * 128,
}

def augment_3d_patch(vol: np.ndarray, lbl: np.ndarray):
    """
    Fast-ish augmentations for 3D patches.
    - vol: float32 (Z,Y,X)
    - lbl: uint8 / int (Z,Y,X) with IGNORE_LABEL=255 supported
    Returns contiguous arrays.
    """
    # Ensure dtypes
    vol = vol.astype(np.float32, copy=False)
    # Keep labels compact; preserve IGNORE_LABEL.
    if lbl.dtype != np.uint8:
        # safe cast if labels are already 0/1/255
        lbl = lbl.astype(np.uint8, copy=False)

    # Random flips
    for axis in range(3):
        if random.random() < AUG["p_flip"]:
            vol = np.flip(vol, axis=axis)
            lbl = np.flip(lbl, axis=axis)

    # Random 90-degree rotation in XY plane
    if AUG["p_rot90"] > 0:
        k = random.randint(0, 3)
        if k:
            vol = np.rot90(vol, k=k, axes=(1, 2))
            lbl = np.rot90(lbl, k=k, axes=(1, 2))

    # Intensity scaling + shift
    if random.random() < AUG["p_intensity"]:
        scale = random.uniform(0.85, 1.15)
        shift = random.uniform(-0.1, 0.1)
        vol = vol * np.float32(scale) + np.float32(shift)

    # Gamma correction
    if random.random() < AUG["p_gamma"]:
        v_min = float(vol.min())
        v_max = float(vol.max())
        if v_max - v_min > 1e-8:
            vol_01 = (vol - v_min) / np.float32(v_max - v_min + 1e-8)
            gamma = random.uniform(0.7, 1.5)
            vol_01 = np.power(np.clip(vol_01, 1e-7, None), np.float32(gamma)).astype(np.float32)
            vol = vol_01 * np.float32(v_max - v_min) + np.float32(v_min)

    # Gaussian noise
    if random.random() < AUG["p_noise"]:
        sigma = random.uniform(0.01, 0.05)
        vol = vol + (np.random.normal(0, sigma, vol.shape).astype(np.float32))

    # Gaussian blur (float32)
    if random.random() < AUG["p_blur"]:
        sigma_blur = float(random.uniform(0.3, 1.0))
        vol = ndi.gaussian_filter(vol.astype(np.float32, copy=False), sigma=sigma_blur).astype(np.float32)

    # Elastic deformation (EXPENSIVE): only on small patches
    if random.random() < AUG["p_elastic"] and vol.size <= AUG["elastic_max_vox"]:
        sigma_el = float(random.uniform(3.0, 5.0))
        alpha_el = float(random.uniform(20.0, 40.0))
        shape = vol.shape

        dx = ndi.gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma_el) * np.float32(alpha_el)
        dy = ndi.gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma_el) * np.float32(alpha_el)
        dz = ndi.gaussian_filter(np.random.randn(*shape).astype(np.float32), sigma_el) * np.float32(alpha_el)

        z, y, x = np.meshgrid(
            np.arange(shape[0], dtype=np.float32),
            np.arange(shape[1], dtype=np.float32),
            np.arange(shape[2], dtype=np.float32),
            indexing="ij",
        )

        coords = [
            np.clip(z + dz, 0, shape[0] - 1),
            np.clip(y + dy, 0, shape[1] - 1),
            np.clip(x + dx, 0, shape[2] - 1),
        ]

        vol = ndi.map_coordinates(vol, coords, order=1, mode="reflect").astype(np.float32)

        # Labels: nearest-neighbor + IGNORE outside
        lbl = ndi.map_coordinates(lbl, coords, order=0, mode="constant", cval=np.uint8(IGNORE_LABEL)).astype(np.uint8)

    return np.ascontiguousarray(vol), np.ascontiguousarray(lbl)

print("[OK] Augmentation ready (time-safe nnUNet-ish)")
HB.progress()

In [ ]:
# ============================================================
# CELL 5: ResidualEncoderUNet (nnUNet-style, 6-stage) — hardened
# ============================================================

class ResidualBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        residual = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)


class EncoderStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks, stride=(1, 1, 1)):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=list(stride), padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.blocks = nn.Sequential(*[ResidualBlock3D(out_ch) for _ in range(n_blocks)])

    def forward(self, x):
        return self.blocks(self.initial(x))


class DecoderStage(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, upsample_stride=(2, 2, 2)):
        super().__init__()
        ks = list(upsample_stride)
        self.upsample = nn.ConvTranspose3d(in_ch, in_ch, kernel_size=ks, stride=ks, bias=False)
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ResidualEncoderUNet(nn.Module):
    def __init__(
        self,
        in_ch=1,
        num_classes=2,
        features=(32, 64, 128, 256, 320, 320),
        blocks_per_stage=(1, 3, 4, 6, 6, 6),
        strides=((1,1,1), (2,2,2), (2,2,2), (2,2,2), (2,2,2), (2,2,2)),
        grad_ckpt_stages=None,
    ):
        super().__init__()
        self.grad_ckpt_stages = set(grad_ckpt_stages or [])
        self.features = features
        self.blocks_per_stage = blocks_per_stage
        self.strides = [tuple(s) for s in strides]

        n_stages = len(features)

        self.encoders = nn.ModuleList()
        for i in range(n_stages):
            in_c = in_ch if i == 0 else features[i - 1]
            self.encoders.append(EncoderStage(in_c, features[i], blocks_per_stage[i], stride=self.strides[i]))

        # Decoder: undo encoder strides in reverse order (matches inference notebook)
        self.decoders = nn.ModuleList()
        for i in range(n_stages - 2, -1, -1):
            up_stride = self.strides[i + 1]  # undo the stride used to go from stage i -> i+1
            self.decoders.append(DecoderStage(features[i + 1], features[i], features[i], upsample_stride=up_stride))

        # Deep supervision heads (optional)
        self.seg_heads = nn.ModuleList()
        for i in range(n_stages - 1):
            self.seg_heads.append(nn.Conv3d(features[i], num_classes, 1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        assert x.ndim == 5, f"Expected 5D (N,C,Z,Y,X), got {x.shape}"

        skips = []
        for i, enc in enumerate(self.encoders):
            if self.training and i in self.grad_ckpt_stages:
                x = grad_checkpoint(enc, x, use_reentrant=False)
            else:
                x = enc(x)
            skips.append(x)

        x = skips[-1]
        decoder_outputs = []
        for i, dec in enumerate(self.decoders):
            x = dec(x, skips[len(skips) - 2 - i])
            decoder_outputs.append(x)

        # decoder_outputs[-1] is finest (features[0])
        out = self.seg_heads[0](decoder_outputs[-1])

        if deep_supervision and self.training:
            # upsample all decoder heads to finest
            ds = [out]
            target = out.shape[2:]
            # next heads correspond to coarser decoder outputs
            for j in range(1, len(decoder_outputs)):
                # map j -> decoder output index from the end
                feat = decoder_outputs[-1 - j]
                lg = self.seg_heads[j](feat)
                lg = F.interpolate(lg, size=target, mode="trilinear", align_corners=False)
                ds.append(lg)
            return tuple(ds)

        return out


# -----------------------------
# Sanity check + patch auto-fit
# -----------------------------
hard_budget_check("Arch sanity start", min_remaining_min=10)

def _dry_run_select_patch():
    """
    Try PATCH_CANDIDATES from large->small.
    Run a tiny forward+backward to catch OOM early.
    Sets global PATCH_SIZE.
    """
    global PATCH_SIZE
    PATCH_SIZE = None
    for ps in PATCH_CANDIDATES:
        print(f"[DRYRUN] Trying patch {ps} ...")
        try:
            m = ResidualEncoderUNet(
                in_ch=1, num_classes=NUM_CLASSES,
                features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE,
                strides=[tuple(s) for s in STRIDES],
                grad_ckpt_stages=GRAD_CKPT_STAGES,
            ).to(DEVICE).train()

            opt = torch.optim.SGD(m.parameters(), lr=0.01)
            x = torch.zeros((1, 1, ps[0], ps[1], ps[2]), device=DEVICE)
            y = torch.zeros((1, ps[0], ps[1], ps[2]), dtype=torch.long, device=DEVICE)

            with _autocast_ctx():
                out = m(x)
                loss = F.cross_entropy(out, y, ignore_index=IGNORE_LABEL)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward() if AMP_ENABLED else loss.backward()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            PATCH_SIZE = ps
            print(f"[OK] Selected PATCH_SIZE={PATCH_SIZE}")
            del m, opt, x, y, out, loss
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            return
        except Exception as e:
            # cleanup on fail
            try:
                del m, opt, x, y, out, loss
            except Exception:
                pass
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            if is_cuda_oom(e):
                print(f"[OOM] Patch {ps} too large.")
                continue
            raise

    raise RuntimeError("[FATAL] All PATCH_CANDIDATES OOM. Add smaller patch sizes or reduce FEATURES.")

# Build a model for param count print
_tmp = ResidualEncoderUNet(
    in_ch=1, num_classes=NUM_CLASSES,
    features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE,
    strides=[tuple(s) for s in STRIDES],
    grad_ckpt_stages=GRAD_CKPT_STAGES,
)
n_params = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] ResidualEncoderUNet: {n_params:.2f}M parameters")
del _tmp; gc.collect()

_dry_run_select_patch()
log_resources("After patch auto-fit")
HB.progress()

In [ ]:
# ============================================================
# CELL 6 (optional extra logging))
# ============================================================
print(f"[OK] PATCH_CANDIDATES = {PATCH_CANDIDATES}")
TRAIN_PATCH = PATCH_SIZE
print(f"[OK] TRAIN_PATCH locked to {TRAIN_PATCH}")
hard_budget_check("Patch size locked", min_remaining_min=10)
log_resources("After patch lock")
HB.progress()

In [ ]:
# ============================================================
# CELL 7: Loss Functions (CE + Dice + MedialSurfaceRecall) — hardened
# ============================================================

class DiceLossFG(nn.Module):
    """
    Foreground-only soft Dice loss with IGNORE_LABEL support.
    Much cheaper than full one-hot Dice.
    """
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        # logits: (B, C, D, H, W)
        # target: (B, D, H, W)
        valid = (target != IGNORE_LABEL)

        probs_fg = torch.softmax(logits, dim=1)[:, 1]  # (B,D,H,W)
        target_fg = (target == 1).float()

        probs_fg = probs_fg * valid
        target_fg = target_fg * valid

        dims = (1, 2, 3)
        inter = (probs_fg * target_fg).sum(dims)
        union = probs_fg.sum(dims) + target_fg.sum(dims)

        dice = 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)
        return dice.mean()


def compute_medial_surface_np(label_3d: np.ndarray) -> np.ndarray:
    """
    Fast 3D medial surface via distance transform ridge detection.
    Returns float32 mask (0/1).
    """
    fg = (label_3d == 1)
    if not fg.any():
        return np.zeros(label_3d.shape, dtype=np.float32)

    dist = ndi.distance_transform_edt(fg)
    local_max = ndi.maximum_filter(dist, size=3)
    medial = (dist > 0) & (dist >= local_max - 1e-6)
    return medial.astype(np.float32)


class MedialSurfaceRecallLoss(nn.Module):
    """
    Encourages recall on medial surface voxels.
    Computes per-batch to keep gradients stable.
    """
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, medial_mask):
        # logits: (B,C,D,H,W)
        # medial_mask: (B,D,H,W) float {0,1}
        probs_fg = torch.softmax(logits, dim=1)[:, 1]

        dims = (1, 2, 3)
        inter = (probs_fg * medial_mask).sum(dims)
        denom = medial_mask.sum(dims) + self.smooth

        recall = inter / denom
        return (1.0 - recall).mean()


class ModelALoss(nn.Module):
    """
    CE + Dice(FG) + MedialSurfaceRecall
    Supports deep supervision (tuple of logits).
    """
    def __init__(self, w_ce=1.0, w_dice=1.0, w_medial=0.5):
        super().__init__()
        self.w_ce = w_ce
        self.w_dice = w_dice
        self.w_medial = w_medial

        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE_LABEL)
        self.dice = DiceLossFG()
        self.medial = MedialSurfaceRecallLoss()

        # nnUNet-style deep supervision weights (finest → coarsest)
        self.ds_weights = [1.0, 0.5, 0.25, 0.125, 0.0625]

    def _loss_single(self, logits, target, medial_mask):
        ce = self.ce(logits, target)
        dice = self.dice(logits, target)
        total = self.w_ce * ce + self.w_dice * dice

        if medial_mask is not None and medial_mask.sum() > 0:
            total = total + self.w_medial * self.medial(logits, medial_mask)

        return total

    def forward(self, logits, target, medial_mask=None):
        # Deep supervision support
        if isinstance(logits, (tuple, list)):
            total = 0.0
            for i, lg in enumerate(logits):
                if i >= len(self.ds_weights):
                    break
                total = total + self.ds_weights[i] * self._loss_single(
                    lg, target, medial_mask
                )
        else:
            total = self._loss_single(logits, target, medial_mask)

        # NaN/Inf guard without breaking graph
        if not torch.isfinite(total):
            print("[WARN] Non-finite loss detected — zeroing loss safely.")
            return total * 0.0

        return total


print("[OK] Loss functions ready (hardened: CE + DiceFG + MedialSurfaceRecall)")
HB.progress()

In [ ]:
# ============================================================
# CELL 8: Dataset (A) — Boundary+FG+BG tri-sampler (ensemble-grade)
# ============================================================
import numpy as np
import random
import scipy.ndimage as ndi
from torch.utils.data import Dataset

P_MEDIAL = 0.60
IGNORE_LABEL = 255

# Sampling mix (sum <= 1; remainder falls to BG)
P_BOUNDARY = 0.45   # highest ROI for surface dice
P_FG       = 0.40   # recall
P_BG       = 0.15   # precision / anti-blob

FG_MIN_VOX = 64
MAX_TRIES  = 48

def _safe_coords(mask: np.ndarray):
    coords = np.argwhere(mask)
    if coords.size == 0:
        return None
    return coords

def _pick_center(coords, shape):
    # coords: (N,3) in (d,h,w)
    c = coords[random.randrange(len(coords))]
    D,H,W = shape
    d = int(np.clip(c[0], 0, D-1))
    h = int(np.clip(c[1], 0, H-1))
    w = int(np.clip(c[2], 0, W-1))
    return d,h,w

class VesuviusPatchDataset(Dataset):
    """
    Cached volume + cached ROI coords:
      - fg coords (label==1)
      - boundary coords (morphological gradient band)
      - bg coords (valid==1 & fg==0)
    Tri-sampling prevents missing thin sheets and improves topology consistency.
    """
    def __init__(self, vol_infos, patch_size=(192,192,192),
                 num_iterations=250, augment=True, p_medial=P_MEDIAL,
                 p_boundary=P_BOUNDARY, p_fg=P_FG, p_bg=P_BG,
                 boundary_dilate=2):
        self.vol_infos = vol_infos
        self.ps = tuple(patch_size)
        self.length = int(num_iterations)
        self.augment = bool(augment)
        self.p_medial = float(p_medial)

        self.p_boundary = float(p_boundary)
        self.p_fg = float(p_fg)
        self.p_bg = float(p_bg)
        self.boundary_dilate = int(boundary_dilate)

        self._cached_idx = None
        self._cached_vol = None
        self._cached_lbl = None

        # Cached coordinate pools for current volume
        self._fg_coords = None
        self._bd_coords = None
        self._bg_coords = None

        print(f"[DATASET-A] vols={len(vol_infos)} iters/epoch={self.length} patch={self.ps} "
              f"mix(boundary/fg/bg)={self.p_boundary:.2f}/{self.p_fg:.2f}/{self.p_bg:.2f}")

    def __len__(self):
        return self.length

    def _load_volume(self, idx):
        vid, img_path, lbl_path = self.vol_infos[idx % len(self.vol_infos)]
        if self._cached_idx == vid and self._cached_vol is not None:
            return self._cached_vol, self._cached_lbl

        vol = read_tif_volume(img_path).astype(np.float32)
        lbl = read_tif_volume(lbl_path).astype(np.uint8)

        # Normalize only on nonzero voxels (avoid background bias)
        m = vol > 0
        if m.any():
            mu = float(vol[m].mean()); sd = float(vol[m].std() + 1e-6)
            vol[m] = (vol[m] - mu) / sd
        vol[~m] = 0.0

        # Fix label semantics
        lbl = lbl.copy()
        lbl[lbl == 2] = IGNORE_LABEL

        # Build valid/fg/boundary/bg pools (cached)
        valid = (lbl != IGNORE_LABEL)
        fg = valid & (lbl == 1)

        # Boundary band from fg morphological gradient
        if fg.any():
            dil = ndi.binary_dilation(fg, iterations=self.boundary_dilate)
            ero = ndi.binary_erosion(fg, iterations=1)
            bd = (dil ^ ero) & valid
        else:
            bd = np.zeros_like(fg, dtype=bool)

        bg = valid & (~fg)

        self._fg_coords = _safe_coords(fg)
        self._bd_coords = _safe_coords(bd)
        self._bg_coords = _safe_coords(bg)

        self._cached_idx = vid
        self._cached_vol = vol
        self._cached_lbl = lbl
        return vol, lbl

    def _crop_around(self, vol, lbl, center):
        D,H,W = vol.shape
        pd,ph,pw = self.ps
        cd,ch,cw = center

        d0 = int(np.clip(cd - pd//2, 0, max(0, D - pd)))
        h0 = int(np.clip(ch - ph//2, 0, max(0, H - ph)))
        w0 = int(np.clip(cw - pw//2, 0, max(0, W - pw)))

        pv = vol[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        pl = lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        return pv, pl

    def __getitem__(self, i):
        # Choose volume uniformly (good generalization)
        vol_idx = random.randrange(len(self.vol_infos))
        vol, lbl = self._load_volume(vol_idx)

        r = random.random()
        mode = "bg"
        coords = self._bg_coords

        if r < self.p_boundary and self._bd_coords is not None:
            mode = "boundary"
            coords = self._bd_coords
        elif r < (self.p_boundary + self.p_fg) and self._fg_coords is not None:
            mode = "fg"
            coords = self._fg_coords
        elif self._bg_coords is not None:
            mode = "bg"
            coords = self._bg_coords

        # Fallback: random crop if pools are empty
        if coords is None:
            D,H,W = vol.shape
            pd,ph,pw = self.ps
            d0 = random.randint(0, max(0, D - pd))
            h0 = random.randint(0, max(0, H - ph))
            w0 = random.randint(0, max(0, W - pw))
            pv = vol[d0:d0+pd, h0:h0+ph, w0:w0+pw]
            pl = lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        else:
            center = _pick_center(coords, vol.shape)
            pv, pl = self._crop_around(vol, lbl, center)

        if self.augment:
            pv, pl = augment_3d_patch(pv, pl)

        if random.random() < self.p_medial:
            pl_clean = pl.copy()
            pl_clean[pl_clean == IGNORE_LABEL] = 0
            medial = compute_medial_surface_np(pl_clean)
        else:
            medial = np.zeros_like(pl, dtype=np.float32)

        x = torch.from_numpy(np.ascontiguousarray(pv[None])).float()
        y = torch.from_numpy(np.ascontiguousarray(pl)).long()
        m = torch.from_numpy(np.ascontiguousarray(medial)).float()

        return x, y, m

print("[OK] Dataset A ready (boundary/fg/bg tri-sampler)")
hard_budget_check("Dataset defined", min_remaining_min=10)
HB.progress()

In [ ]:
# ============================================================
# CELL 8.5: SANITY PROBE (Model A) — run BEFORE training
# ============================================================
import numpy as np
import scipy.ndimage as ndi

# IMPORTANT: Dataset A does NOT take fg_rate. It uses p_boundary/p_fg/p_bg.
train_dataset = VesuviusPatchDataset(
    train_vols,
    patch_size=TRAIN_PATCH,
    num_iterations=min(32, NUM_ITERATIONS),
    augment=True,
    p_medial=0.0,  # speed: avoid extra medial work for probe
)

print("[OK] Created probe train_dataset for sanity-check:", type(train_dataset))

def _describe_mask(m, name="mask"):
    m = (m > 0).astype(np.uint8)
    fg = int(m.sum())
    tot = int(m.size)
    fg_pct = 100.0 * fg / max(1, tot)
    struct26 = ndi.generate_binary_structure(3, 3)
    cc, ncc = ndi.label(m, structure=struct26)
    largest_pct = 0.0
    if ncc > 0 and fg > 0:
        sizes = np.bincount(cc.ravel())[1:]
        largest = int(sizes.max()) if sizes.size else 0
        largest_pct = 100.0 * largest / max(1, fg)
    print(f"[{name}] FG%={fg_pct:.4f}% | nCC(26)={ncc} | largest_share={largest_pct:.2f}%")

def _extract_y(sample):
    if isinstance(sample, (tuple, list)):
        y = sample[1]
    elif isinstance(sample, dict):
        y = sample.get("label", sample.get("y", None))
    else:
        y = None
    if y is None:
        raise RuntimeError("Could not extract label/mask from dataset sample.")
    if hasattr(y, "detach"):
        y = y.detach().cpu().numpy()
    return y

for i in range(5):
    x, y = train_dataset[i]
    y_np = _extract_y((x, y))
    _describe_mask(y_np, name=f"train[{i}]")

print("[OK] Sanity probe complete.")

In [ ]:
# ============================================================
# SHARED VALIDATION UTILITY (Insert between Cell 9 and Cell 10)
# Full-volume leaderboard-aligned validation:
# - SurfaceDice@tau (tau=2.0)
# - VOI_score (alpha=0.3) via 26-connectivity CC labelings
# - Topology proxies (splits + cavities) for TopoScore correlation
# ============================================================

import numpy as np
import torch
import scipy.ndimage as ndi

VAL_TAU = 2.0
VOI_ALPHA = 0.3
CC_CONN = 3  # 26-connectivity via generate_binary_structure(3,3)

def _softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x, dtype=np.float32)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-8)

def _gaussian_weight_map(shape_zyx, sigma_scale=0.125):
    D, H, W = shape_zyx
    zz = np.linspace(-1, 1, D, dtype=np.float32)
    yy = np.linspace(-1, 1, H, dtype=np.float32)
    xx = np.linspace(-1, 1, W, dtype=np.float32)
    Z, Y, X = np.meshgrid(zz, yy, xx, indexing="ij")
    r2 = Z*Z + Y*Y + X*X
    sigma2 = (sigma_scale ** 2)
    w = np.exp(-0.5 * r2 / max(sigma2, 1e-6)).astype(np.float32)
    return w / (w.max() + 1e-8)

@torch.no_grad()
def svu_sliding_window_logits(model, vol_zyx, roi_size=(64,192,192), overlap=0.35, num_classes=2):
    """
    vol_zyx: np.ndarray float32 (D,H,W)
    returns logits_zyxc: np.ndarray float32 (D,H,W,C)
    """
    model.eval()
    D, H, W = vol_zyx.shape
    rz, ry, rx = roi_size

    sz = max(1, int(rz * (1.0 - overlap)))
    sy = max(1, int(ry * (1.0 - overlap)))
    sx = max(1, int(rx * (1.0 - overlap)))

    z_starts = list(range(0, max(D - rz, 0) + 1, sz)) or [0]
    y_starts = list(range(0, max(H - ry, 0) + 1, sy)) or [0]
    x_starts = list(range(0, max(W - rx, 0) + 1, sx)) or [0]
    if D > rz and z_starts[-1] != D - rz: z_starts.append(D - rz)
    if H > ry and y_starts[-1] != H - ry: y_starts.append(H - ry)
    if W > rx and x_starts[-1] != W - rx: x_starts.append(W - rx)

    out = np.zeros((D, H, W, num_classes), dtype=np.float32)
    wsum = np.zeros((D, H, W, 1), dtype=np.float32)

    w_patch = _gaussian_weight_map((min(rz, D), min(ry, H), min(rx, W)))[..., None]

    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                z1 = min(z0 + rz, D); y1 = min(y0 + ry, H); x1 = min(x0 + rx, W)
                patch = vol_zyx[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)

                padz = rz - patch.shape[0]; pady = ry - patch.shape[1]; padx = rx - patch.shape[2]
                if padz or pady or padx:
                    patch = np.pad(patch, ((0,padz),(0,pady),(0,padx)), mode="reflect")

                inp = torch.from_numpy(patch[None, None]).to(DEVICE, non_blocking=True)  # (1,1,rz,ry,rx)
                logits = model(inp)  # (1,C,rz,ry,rx)
                logits = logits.float().cpu().numpy()[0].transpose(1,2,3,0)  # (rz,ry,rx,C)

                logits = logits[:(z1-z0), :(y1-y0), :(x1-x0), :]
                wp = w_patch[:(z1-z0), :(y1-y0), :(x1-x0), :]

                out[z0:z1, y0:y1, x0:x1, :] += logits * wp
                wsum[z0:z1, y0:y1, x0:x1, :] += wp

    out = out / np.clip(wsum, 1e-6, None)
    return out

def svu_hysteresis_bin(prob, tl, th):
    strong = prob >= th
    weak = prob >= tl
    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    lbl, n = ndi.label(weak, structure=struct26)
    if n == 0:
        return np.zeros_like(prob, dtype=np.uint8)
    strong_ids = np.unique(lbl[strong])
    strong_ids = strong_ids[strong_ids != 0]
    keep = np.isin(lbl, strong_ids)
    return keep.astype(np.uint8)

def svu_surface_dice_at_tau(pred_bin, gt_bin, spacing=(1.0,1.0,1.0), tau=2.0):
    pred = (pred_bin > 0)
    gt = (gt_bin > 0)

    if pred.sum() == 0 and gt.sum() == 0: return 1.0
    if (pred.sum() == 0) ^ (gt.sum() == 0): return 0.0

    st = ndi.generate_binary_structure(3, 1)
    pred_s = pred & ~ndi.binary_erosion(pred, structure=st, iterations=1, border_value=0)
    gt_s   = gt   & ~ndi.binary_erosion(gt,   structure=st, iterations=1, border_value=0)

    dt_gt = ndi.distance_transform_edt(~gt_s, sampling=spacing)
    dt_pr = ndi.distance_transform_edt(~pred_s, sampling=spacing)

    p2g = (dt_gt[pred_s] <= tau).mean() if pred_s.any() else 1.0
    g2p = (dt_pr[gt_s]   <= tau).mean() if gt_s.any() else 1.0
    return float(0.5 * (p2g + g2p))

def svu_voi_score(pred_bin, gt_bin, alpha=0.3):
    pred = (pred_bin > 0)
    gt   = (gt_bin > 0)

    if pred.sum() == 0 and gt.sum() == 0: return 1.0
    if (pred.sum() == 0) ^ (gt.sum() == 0): return 0.0

    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    pred_cc, _ = ndi.label(pred, structure=struct26)
    gt_cc, _   = ndi.label(gt,   structure=struct26)

    u = pred | gt
    p = pred_cc[u].astype(np.int64, copy=False)
    g = gt_cc[u].astype(np.int64, copy=False)

    _, p = np.unique(p, return_inverse=True)
    _, g = np.unique(g, return_inverse=True)

    n = p.size
    if n == 0: return 1.0

    Pg = p.max() + 1
    Gg = g.max() + 1
    idx = p * Gg + g
    c = np.bincount(idx, minlength=Pg*Gg).astype(np.float64).reshape(Pg, Gg)

    P = c / n
    p_m = P.sum(axis=1, keepdims=True)
    g_m = P.sum(axis=0, keepdims=True)

    eps = 1e-12
    voi_split = -np.sum(P * (np.log(P + eps) - np.log(p_m + eps)))
    voi_merge = -np.sum(P * (np.log(P + eps) - np.log(g_m + eps)))
    voi_total = float(voi_split + voi_merge)

    return float(1.0 / (1.0 + alpha * voi_total))

def svu_topo_proxies(pred_bin, gt_bin):
    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    pred = (pred_bin > 0)
    gt   = (gt_bin > 0)

    _, pred_n = ndi.label(pred, structure=struct26)
    _, gt_n   = ndi.label(gt,   structure=struct26)

    fg = pred | gt
    cavities = 0
    if fg.any():
        coords = np.argwhere(fg)
        z0,y0,x0 = coords.min(axis=0); z1,y1,x1 = coords.max(axis=0) + 1
        pad = 5
        z0 = max(0, z0-pad); y0=max(0,y0-pad); x0=max(0,x0-pad)
        z1 = min(fg.shape[0], z1+pad); y1=min(fg.shape[1], y1+pad); x1=min(fg.shape[2], x1+pad)

        roi_fg = fg[z0:z1, y0:y1, x0:x1]
        roi_bg = ~roi_fg
        bg_cc, bg_n = ndi.label(roi_bg, structure=struct26)

        touch = np.zeros(bg_n + 1, dtype=bool)
        touch[np.unique(bg_cc[0,:,:])] = True
        touch[np.unique(bg_cc[-1,:,:])] = True
        touch[np.unique(bg_cc[:,0,:])] = True
        touch[np.unique(bg_cc[:,-1,:])] = True
        touch[np.unique(bg_cc[:,:,0])] = True
        touch[np.unique(bg_cc[:,:,-1])] = True

        cavities = sum((not touch[i]) for i in range(1, bg_n+1))

    return {"pred_components": int(pred_n), "gt_components": int(gt_n), "cavities_proxy": int(cavities)}

@torch.no_grad()
def svu_validate_full_volume_case(model, img_path, lbl_path, roi_size, overlap, tl, th, spacing=(1.0,1.0,1.0)):
    vol = read_tif_volume(img_path).astype(np.float32)
    gt  = read_tif_volume(lbl_path).astype(np.uint8)

    gt[gt == 2] = IGNORE_LABEL
    valid = (gt != IGNORE_LABEL)
    gt_bin = ((gt == 1) & valid).astype(np.uint8)

    logits = svu_sliding_window_logits(model, vol, roi_size=roi_size, overlap=overlap, num_classes=NUM_CLASSES)
    prob = _softmax_np(logits, axis=-1)[..., 1].astype(np.float32)

    pred_bin = svu_hysteresis_bin(prob, tl, th)
    pred_bin = (pred_bin & valid).astype(np.uint8)

    sd  = svu_surface_dice_at_tau(pred_bin, gt_bin, spacing=spacing, tau=VAL_TAU)
    voi = svu_voi_score(pred_bin, gt_bin, alpha=VOI_ALPHA)
    tp  = svu_topo_proxies(pred_bin, gt_bin)

    return {"surface_dice_tau": float(sd), "voi_score": float(voi), **tp}

def svu_combined_val_score(m):
    sd  = m["surface_dice_tau"]
    voi = m["voi_score"]

    split_pen = min(1.0, abs(m["pred_components"] - m["gt_components"]) / max(1, m["gt_components"]))
    split_proxy = 1.0 - split_pen

    cav = m["cavities_proxy"]
    cav_proxy = 1.0 / (1.0 + 0.25 * cav)

    topo_proxy = 0.5 * split_proxy + 0.5 * cav_proxy
    return float(0.35*sd + 0.35*voi + 0.30*topo_proxy)

print("[OK] Shared full-volume validation utilities ready")

In [ ]:
# ============================================================
# CELL 9: Training Infrastructure (Model A) — FIXED VALIDATION + BEST BY VAL
# ============================================================
from contextlib import contextmanager
import numpy as np
import torch
from torch.utils.data import DataLoader

@contextmanager
def _autocast_ctx():
    if AMP_ENABLED and DEVICE.type == "cuda":
        with autocast("cuda", dtype=AMP_DTYPE):
            yield
    else:
        yield

def save_ckpt_safe(path, obj):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

# ----------------------------
# EMA (CPU shadow weights)
# ----------------------------
class EMAModel:
    def __init__(self, model, decay=0.9999):
        self.decay = float(decay)
        self.shadow = {k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in msd.items():
            v_cpu = v.detach().float().cpu()
            self.shadow[k].mul_(self.decay).add_(v_cpu, alpha=1.0 - self.decay)

# ----------------------------
# Fixed validation patch cache (honest + stable)
# ----------------------------
def _softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x, dtype=np.float32)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-8)

def build_fixed_val_cache(val_vols, n_patches=512, seed=123):
    rng = np.random.RandomState(seed)
    ds = VesuviusPatchDataset(
        val_vols,
        patch_size=TRAIN_PATCH,
        num_iterations=n_patches,
        augment=False,
        p_medial=0.0,   # keep val stable/cheap
    )
    cache = []
    for i in range(n_patches):
        x, y = ds[i]
        # x: torch tensor (1,D,H,W) or (D,H,W) depending on your dataset output
        if x.ndim == 3:
            x = x[None]
        if y.ndim == 3:
            y = y[None]
        cache.append((x.contiguous(), y.contiguous()))
    return cache

@torch.no_grad()
def validate_on_cache(model, cache):
    model.eval()
    dices = []
    for x, y in cache:
        inp = x[None].to(DEVICE, non_blocking=True)  # (1,1,D,H,W)
        with _autocast_ctx():
            out = model(inp)  # (1,C,D,H,W)
        out = out.float().cpu().numpy()[0].transpose(1,2,3,0)  # (D,H,W,C)
        p = _softmax_np(out, axis=-1)[..., 1]
        pred = (p >= 0.5).astype(np.uint8)
        gt = (y.numpy()[0] > 0.5).astype(np.uint8)
        inter = (pred & gt).sum()
        denom = pred.sum() + gt.sum()
        dice = (2.0 * inter / denom) if denom > 0 else 1.0
        dices.append(float(dice))
    model.train()
    return float(np.mean(dices))

# ----------------------------
# Train
# ----------------------------
def train_model():
    print(f"\n{'='*60}")
    print("[TRAIN] Model A: ResidualEncoderUNet")
    print(f"  PATCH={TRAIN_PATCH} | LR={INITIAL_LR} | GRAD_ACCUM={GRAD_ACCUM} | AMP={AMP_ENABLED}")
    print(f"  Budget={MAX_TRAIN_HOURS:.2f}h | iters/epoch={NUM_ITERATIONS} | val vols={len(val_vols)}")
    print(f"{'='*60}\n")

    hard_budget_check("Train init", min_remaining_min=15)
    log_resources("Train init")

    model = ResidualEncoderUNet(
        in_ch=1, num_classes=NUM_CLASSES,
        features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE,
        strides=[tuple(s) for s in STRIDES],
        grad_ckpt_stages=GRAD_CKPT_STAGES,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=1e-4)
    scaler = GradScaler("cuda") if (DEVICE.type == "cuda" and AMP_ENABLED) else None

    ema = EMAModel(model, decay=EMA_DECAY)

    # Dataset A does NOT take fg_rate:
    train_ds = VesuviusPatchDataset(
        train_vols,
        patch_size=TRAIN_PATCH,
        num_iterations=NUM_ITERATIONS,
        augment=True,
    )
    loader = DataLoader(train_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    # Build a fixed val cache once
    VAL_PATCHES = int(min(1024, max(256, NUM_ITERATIONS // 4)))
    val_cache = build_fixed_val_cache(val_vols, n_patches=VAL_PATCHES, seed=SEED + 999)
    print(f"[VAL] Fixed cache built: {len(val_cache)} patches")

    best_val = -1.0
    losses = []

    train_start = time.time()
    for epoch in range(TARGET_EPOCHS):
        # budget guard
        if (time.time() - train_start) / 3600.0 > MAX_TRAIN_HOURS:
            print("[BUDGET] Max train hours reached. Stopping.")
            break

        model.train()
        running = 0.0
        n = 0

        optimizer.zero_grad(set_to_none=True)

        for it in range(NUM_ITERATIONS):
            x, y = train_ds[it]
            if x.ndim == 3: x = x[None]
            if y.ndim == 3: y = y[None]
            inp = x[None].to(DEVICE, non_blocking=True)  # (1,1,D,H,W)
            tgt = y[None].to(DEVICE, non_blocking=True)  # (1,1,D,H,W)

            with torch.cuda.amp.autocast(enabled=(scaler is not None), dtype=AMP_DTYPE):
                out = model(inp)  # (1,C,D,H,W)
                loss = criterion(out, tgt)

            if scaler is not None:
                scaler.scale(loss / GRAD_ACCUM).backward()
            else:
                (loss / GRAD_ACCUM).backward()

            if ((it + 1) % GRAD_ACCUM) == 0:
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

                ema.update(model)

            running += float(loss.detach().cpu().item())
            n += 1

            del inp, tgt, out, loss
            if DEVICE.type == "cuda" and ((it + 1) % 64 == 0):
                torch.cuda.empty_cache()

        avg_loss = running / max(n, 1)
        losses.append(avg_loss)

        # Apply EMA weights for val
        backup = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(ema.shadow, strict=True)

        val_dice = validate_on_cache(model, val_cache)
        print(f"[E{epoch+1:02d}] train_loss={avg_loss:.4f} | val_dice@0.5={val_dice:.4f}")

        # --- Full-volume leaderboard-aligned check (every VAL_EVERY epochs) ---
        VAL_EVERY = 2
        VAL_N_FULL = 1
        FULL_OVERLAP = 0.35
        FULL_ROI = (TRAIN_PATCH[0], TRAIN_PATCH[1], TRAIN_PATCH[2])  # same as training patch
        TL_TH = (0.55, 0.85)

        if ((epoch + 1) % VAL_EVERY) == 0:
            full_cases = val_vol_infos[:VAL_N_FULL]
            full_metrics = []
            for img_path, _, lbl_path in full_cases:
                m = svu_validate_full_volume_case(
                    model, img_path, lbl_path,
                    roi_size=FULL_ROI, overlap=FULL_OVERLAP,
                    tl=TL_TH[0], th=TL_TH[1],
                    spacing=(1.0,1.0,1.0),
                )   
                full_metrics.append(m)
            mean_combo = float(np.mean([svu_combined_val_score(mm) for mm in full_metrics]))
            print(f"[FULLVAL] mean_combo={mean_combo:.4f} | details={full_metrics[0] if full_metrics else None}")

            if ("best_combo" not in locals()) or (mean_combo > best_combo):
                best_combo = mean_combo
                torch.save(model.state_dict(), f"{CKPT_DIR}/model_a_best_combo.pt")
                print(f"[SAVE] New best COMBO={best_combo:.4f} -> {CKPT_DIR}/model_a_best_combo.pt")
        
        # Save best by VAL
        if val_dice > best_val:
            best_val = val_dice
            torch.save(model.state_dict(), f"{CKPT_DIR}/model_a_best.pt")
            print(f"[SAVE] New best val={best_val:.4f} -> {CKPT_DIR}/model_a_best.pt")

        # Restore training weights and continue
        model.load_state_dict(backup, strict=True)
        del backup
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model, losses

In [ ]:
# ============================================================
# CELL 10: Execute Training (fail-fast, budget-safe)
# ============================================================
hard_budget_check("Training start", min_remaining_min=20)
log_resources("Before train_model")
HB.beat("train_exec", "start")

try:
    model_a, losses_a, val_history_a = train_model()
except Exception as e:
    print(f"[FATAL] Training crashed: {e}")
    # If it was OOM, you already get oom_failfast logs; this is just top-level clarity.
    raise
finally:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()
    log_resources("After train_model cleanup")
    HB.progress()

hard_budget_check("Training complete", min_remaining_min=10)
print("[OK] Training finished and cleaned up")

In [ ]:
# ============================================================
# CELL 11: Threshold Calibration (Model A) — META ONLY (no inference)
# Replaces existing Cell 12 in 1_train_model_a.ipynb
# ============================================================

import os, json
import numpy as np

PP_VERSION = "vesu_pp_v2"
MODEL_TAG = "a"

def fg_fraction_from_lbl(lbl):
    valid = (lbl != IGNORE_LABEL)
    if not valid.any(): return 0.0
    return float(((lbl == 1) & valid).sum() / (valid.sum() + 1e-12))

val_fracs = []
for _, _, lbl_path in val_vol_infos:
    gt = read_tif_volume(lbl_path).astype(np.uint8)
    gt[gt == 2] = IGNORE_LABEL
    val_fracs.append(fg_fraction_from_lbl(gt))

target_fg_med = float(np.median(val_fracs))
target_fg_lo  = float(np.quantile(val_fracs, 0.25))
target_fg_hi  = float(np.quantile(val_fracs, 0.75))

TH_PACK = [
    {"tl": 0.55, "th": 0.85},
    {"tl": 0.50, "th": 0.80},
    {"tl": 0.60, "th": 0.88},
]

meta = {
    "pp_version": PP_VERSION,
    "model_tag": "A",
    "target_fg_med": target_fg_med,
    "target_fg_lo": target_fg_lo,
    "target_fg_hi": target_fg_hi,
    "th_pack": TH_PACK,
}

os.makedirs(CKPT_DIR, exist_ok=True)
meta_path = os.path.join(CKPT_DIR, "model_a_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[OK] Wrote {meta_path}")
print(f"[CAL-A] FG median={target_fg_med:.6f} IQR=({target_fg_lo:.6f},{target_fg_hi:.6f}) | pack={len(TH_PACK)}")

In [ ]:
# ============================================================
# CELL 12: Results & Diagnostic Plots (aligned with calib tl/th)
# ============================================================
import glob

hard_budget_check("Diagnostics start", min_remaining_min=5)
HB.beat("diag", "start")
log_resources("Before diagnostics")

print(f"\n{'='*60}")
print("CHECKPOINTS:")
print(f"{'='*60}")
for pt in sorted(glob.glob(f"{CKPT_DIR}/*.pt")):
    sz = os.path.getsize(pt) / 1e6
    print(f"  {os.path.basename(pt):30s}  ({sz:.1f} MB)")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1) Full-volume Score vs Epoch
    ax = axes[0, 0]
    if val_history_a:
        epochs = [e for e, _ in val_history_a]
        means = [v['mean'] for _, v in val_history_a]
        mins  = [v['min']  for _, v in val_history_a]
        ax.plot(epochs, means, marker='o', markersize=4, label='Mean score')
        ax.plot(epochs, mins,  marker='s', markersize=3, alpha=0.8, label='Min score')
        ax.set_xlabel("Epoch"); ax.set_ylabel("Competition Score")
        ax.set_title("Full-Volume Validation vs Epoch"); ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "No val_history_a", ha="center", va="center")
        ax.set_axis_off()

    # 2) Loss vs Epoch
    ax = axes[0, 1]
    if losses_a:
        ax.plot(list(range(1, len(losses_a)+1)), losses_a, marker='o', markersize=3)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title("Training Loss vs Epoch")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "No losses_a", ha="center", va="center")
        ax.set_axis_off()

    # 3) Calibration heatmap (t_low, t_high) -> mean_score
    ax = axes[1, 0]
    if "calib_results" in globals() and calib_results:
        tls = sorted(list(set([round(float(tl), 3) for tl, _, _, _ in calib_results])))
        ths = sorted(list(set([round(float(th), 3) for _, th, _, _ in calib_results])))

        grid = np.full((len(ths), len(tls)), np.nan, dtype=np.float32)
        for tl, th, mean_s, min_s in calib_results:
            i = ths.index(round(float(th), 3))
            j = tls.index(round(float(tl), 3))
            grid[i, j] = float(mean_s)

        im = ax.imshow(grid, aspect='auto', origin='lower')
        ax.set_xticks(range(len(tls))); ax.set_xticklabels([f"{t:.2f}" for t in tls], rotation=45)
        ax.set_yticks(range(len(ths))); ax.set_yticklabels([f"{t:.2f}" for t in ths])
        ax.set_xlabel("t_low"); ax.set_ylabel("t_high")
        ax.set_title("Calibration: mean score heatmap")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # mark best
        if "best_tl" in globals() and "best_th" in globals():
            bj = tls.index(round(float(best_tl), 3))
            bi = ths.index(round(float(best_th), 3))
            ax.plot([bj], [bi], marker='x', markersize=12)
    else:
        ax.text(0.5, 0.5, "No calib_results", ha="center", va="center")
        ax.set_axis_off()

    # 4) Calibration min-score heatmap
    ax = axes[1, 1]
    if "calib_results" in globals() and calib_results:
        tls = sorted(list(set([round(float(tl), 3) for tl, _, _, _ in calib_results])))
        ths = sorted(list(set([round(float(th), 3) for _, th, _, _ in calib_results])))

        grid = np.full((len(ths), len(tls)), np.nan, dtype=np.float32)
        for tl, th, mean_s, min_s in calib_results:
            i = ths.index(round(float(th), 3))
            j = tls.index(round(float(tl), 3))
            grid[i, j] = float(min_s)

        im = ax.imshow(grid, aspect='auto', origin='lower')
        ax.set_xticks(range(len(tls))); ax.set_xticklabels([f"{t:.2f}" for t in tls], rotation=45)
        ax.set_yticks(range(len(ths))); ax.set_yticklabels([f"{t:.2f}" for t in ths])
        ax.set_xlabel("t_low"); ax.set_ylabel("t_high")
        ax.set_title("Calibration: min score heatmap")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        if "best_tl" in globals() and "best_th" in globals():
            bj = tls.index(round(float(best_tl), 3))
            bi = ths.index(round(float(best_th), 3))
            ax.plot([bj], [bi], marker='x', markersize=12)
    else:
        ax.text(0.5, 0.5, "No calib_results", ha="center", va="center")
        ax.set_axis_off()

    plt.tight_layout()
    out_png = "/kaggle/working/diagnostics.png"
    plt.savefig(out_png, dpi=140)
    print(f"[OK] Diagnostic plots saved to {out_png}")

except Exception as e:
    print(f"[WARN] Plot failed: {e}")
    import traceback; traceback.print_exc()

hard_budget_check("Diagnostics done", min_remaining_min=1)
log_resources("After diagnostics")
HB.progress()

print(f"\n{'='*60}")
bt = f"{float(best_tl):.3f}/{float(best_th):.3f}" if ("best_tl" in globals() and "best_th" in globals()) else "N/A"
print(f"[SUCCESS] Best (t_low/t_high): {bt}")
print(f"[SUCCESS] Download model_a.pt (+ model_a_best.pt if present), upload as Kaggle Dataset")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 12.5: Canonicalize Model A artifacts (ensemble-proof)
#   - Ensures model_a.pt exists AND is saved as {"state_dict": ...}
#   - Keeps best/last naming
#   - Writes model_a_meta.json
# ============================================================
import os, glob, json, shutil, time, torch

os.makedirs(CKPT_DIR, exist_ok=True)

def _copy_atomic(src, dst):
    tmp = dst + ".tmp"
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def _pick_latest_epoch_ckpt():
    pts = sorted(glob.glob(os.path.join(CKPT_DIR, "model_a_epoch*.pt")))
    return pts[-1] if pts else None

def _load_state_dict_any(path):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
        return obj["state_dict"], obj
    # raw state_dict case (e.g. ema.shadow)
    if isinstance(obj, dict) and all(hasattr(v, "shape") for v in obj.values()):
        return obj, {"raw_state_dict": True}
    raise ValueError(f"Unrecognized checkpoint format: {path}")

def _wrap_and_save(path_out, state_dict, epoch=None, extra=None):
    ckpt = {
        "state_dict": state_dict,
        "epoch": None if epoch is None else int(epoch),
        "model_tag": "A",
        "config": {
            "num_classes": int(NUM_CLASSES) if "NUM_CLASSES" in globals() else 2,
            "features": FEATURES if "FEATURES" in globals() else None,
            "blocks_per_stage": BLOCKS_PER_STAGE if "BLOCKS_PER_STAGE" in globals() else None,
            "strides": STRIDES if "STRIDES" in globals() else None,
            "roi": TRAIN_PATCH if "TRAIN_PATCH" in globals() else None,
            "ignore_label": 255,
            "fg_is_label_1": True,
            "label_2_is_ignore": True,
        },
        "extra": extra if isinstance(extra, dict) else {},
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp = path_out + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, path_out)

best_path = os.path.join(CKPT_DIR, "model_a_best.pt")
last_path = os.path.join(CKPT_DIR, "model_a_last.pt")
canon_path = os.path.join(CKPT_DIR, "model_a.pt")
meta_path  = os.path.join(CKPT_DIR, "model_a_meta.json")

latest_epoch = _pick_latest_epoch_ckpt()

picked_last = None
picked_best = None

# 1) last
if latest_epoch is not None:
    picked_last = latest_epoch
    _copy_atomic(latest_epoch, last_path)
elif os.path.exists(best_path):
    picked_last = best_path
    _copy_atomic(best_path, last_path)
else:
    any_pt = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt")))
    if any_pt:
        picked_last = any_pt[-1]
        _copy_atomic(picked_last, last_path)

# 2) best
if os.path.exists(best_path):
    picked_best = best_path
elif picked_last is not None and os.path.exists(picked_last):
    picked_best = picked_last
    _copy_atomic(picked_last, best_path)

# 3) canonical model_a.pt as WRAPPED dict
canon_src = None
if picked_best is not None and os.path.exists(picked_best):
    sd, raw = _load_state_dict_any(picked_best)
    _wrap_and_save(canon_path, sd, epoch=None, extra={"canonical_src": picked_best, **({"raw": raw} if isinstance(raw, dict) else {})})
    canon_src = picked_best
elif os.path.exists(last_path):
    sd, raw = _load_state_dict_any(last_path)
    _wrap_and_save(canon_path, sd, epoch=None, extra={"canonical_src": last_path, **({"raw": raw} if isinstance(raw, dict) else {})})
    canon_src = last_path
else:
    canon_src = None

meta = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "ckpt_dir": CKPT_DIR,
    "model_a_best_exists": bool(os.path.exists(best_path)),
    "model_a_last_exists": bool(os.path.exists(last_path)),
    "model_a_pt_exists": bool(os.path.exists(canon_path)),
    "picked_last_src": picked_last,
    "picked_best_src": picked_best,
    "canonical_src": canon_src,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[ARTIFACT] model_a.pt:      {'OK' if os.path.exists(canon_path) else 'MISSING'} (wrapped)")
print(f"[ARTIFACT] model_a_best.pt: {'OK' if os.path.exists(best_path) else 'MISSING'}")
print(f"[ARTIFACT] model_a_last.pt: {'OK' if os.path.exists(last_path) else 'MISSING'}")
print("[ARTIFACT] model_a_meta.json written")

In [ ]:
# ============================================================
# CELL 13: Results Summary (robust, aligned with fixed CELL 9)
# ============================================================
import os, glob, json

hard_budget_check("Results summary start", min_remaining_min=1)
HB.beat("summary", "start")
log_resources("Before summary")

def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default

def _extract_train_artifacts():
    """
    Supports BOTH styles:
    - New: train_out = {"losses": [...], "val_history": [(epoch, vr), ...], "best_mean": ...}
    - Old: losses_a, val_history_a
    """
    losses = None
    val_history = None
    best_mean = None

    if "train_out" in globals() and isinstance(train_out, dict):
        losses = train_out.get("losses", None)
        val_history = train_out.get("val_history", None)
        best_mean = train_out.get("best_mean", None)

    if losses is None and "losses_a" in globals():
        losses = globals().get("losses_a", None)

    if val_history is None and "val_history_a" in globals():
        val_history = globals().get("val_history_a", None)

    return losses, val_history, best_mean

def _vr_min_from_by_vol(vr):
    """
    Some validators return vr['min']; others return vr['by_vol'] dict.
    This computes a min safely if possible.
    """
    if isinstance(vr, dict):
        if "min" in vr and vr["min"] is not None:
            return _safe_float(vr["min"], default=None)
        byv = vr.get("by_vol", None)
        if isinstance(byv, dict) and len(byv) > 0:
            vals = [_safe_float(v, default=None) for v in byv.values()]
            vals = [v for v in vals if v is not None]
            if len(vals) > 0:
                return min(vals)
    return None

def _vr_mean(vr):
    if isinstance(vr, dict) and "mean" in vr:
        return _safe_float(vr["mean"], default=None)
    return None

# --------------------------------
# Checkpoints
# --------------------------------
print(f"\n{'='*60}")
print("CHECKPOINTS:")
print(f"{'='*60}")
ckpts = sorted(glob.glob(f"{CKPT_DIR}/*.pt"))
if not ckpts:
    print("  (none found)")
else:
    for pt in ckpts:
        sz = os.path.getsize(pt) / 1e6
        print(f"  {os.path.basename(pt):35s} ({sz:6.1f} MB)")

# --------------------------------
# Training summary
# --------------------------------
losses, val_history, best_mean = _extract_train_artifacts()

print(f"\n{'='*60}")
print("TRAINING SUMMARY:")
print(f"{'='*60}")

if losses and isinstance(losses, (list, tuple)) and len(losses) > 0:
    last_loss = _safe_float(losses[-1], default=None)
    best_loss = min([_safe_float(x, default=1e9) for x in losses])
    print(f"  epochs_trained: {len(losses)}")
    if last_loss is not None:
        print(f"  last_loss:      {last_loss:.6f}")
    print(f"  best_loss:      {best_loss:.6f}")
else:
    print("  losses: (missing or empty)")

# --------------------------------
# Validation summary
# --------------------------------
if val_history and isinstance(val_history, (list, tuple)) and len(val_history) > 0:
    means = []
    mins = []

    for e, vr in val_history:
        m = _vr_mean(vr)
        mn = _vr_min_from_by_vol(vr)
        if m is not None:
            means.append((e, m))
        if mn is not None:
            mins.append((e, mn))

    last_epoch, last_vr = val_history[-1]
    last_mean = _vr_mean(last_vr)
    last_min  = _vr_min_from_by_vol(last_vr)

    print(f"  val_checks:     {len(val_history)}")

    if len(means) > 0:
        best_mean_epoch, best_mean_val = max(means, key=lambda x: x[1])
        print(f"  best_mean:      {best_mean_val:.4f} @ epoch {best_mean_epoch}")
    else:
        print("  best_mean:      (not available)")

    if len(mins) > 0:
        best_min_epoch, best_min_val = max(mins, key=lambda x: x[1])
        print(f"  best_min:       {best_min_val:.4f} @ epoch {best_min_epoch}")
    else:
        print("  best_min:       (not available)")

    if last_mean is not None:
        print(f"  last_val_mean:  {last_mean:.4f} @ epoch {last_epoch}")
    else:
        print(f"  last_val_mean:  (not available) @ epoch {last_epoch}")

    if last_min is not None:
        print(f"  last_val_min:   {last_min:.4f}  @ epoch {last_epoch}")
    else:
        print(f"  last_val_min:   (not available) @ epoch {last_epoch}")
else:
    print("  val_history: (missing or empty)")

# --------------------------------
# Calibration summary
# --------------------------------
bt = None
if "best_tl" in globals() and "best_th" in globals():
    bt = (_safe_float(best_tl, 0.50), _safe_float(best_th, 0.75))
elif os.path.exists(f"{CKPT_DIR}/model_a_calib.json"):
    try:
        with open(f"{CKPT_DIR}/model_a_calib.json", "r") as f:
            d = json.load(f)
        bt = (_safe_float(d.get("t_low", 0.50), 0.50), _safe_float(d.get("t_high", 0.75), 0.75))
    except Exception:
        bt = None

if bt is not None:
    print(f"  calib (t_low/t_high): {bt[0]:.3f}/{bt[1]:.3f}")
else:
    print("  calib (t_low/t_high): (not found)")

# --------------------------------
# Artifact recommendation
# --------------------------------
print(f"\n{'='*60}")
print("[ARTIFACTS] Suggested outputs for Kaggle Dataset upload:")
print("  - model_a_best.pt        (EMA weights at best mean val, if created)")
print("  - model_a_epochXX.pt     (optional checkpoints)")
print("  - model_a_calib.json     (thresholds for inference, if you created it)")
print("\n[NOTE] If you want a single canonical file name, copy/symlink:")
print("  - model_a.pt  -> choose model_a_best.pt (or latest epoch) and rename.")
print(f"{'='*60}")

HB.progress()
hard_budget_check("Results summary done", min_remaining_min=0)